# Qwen3-32B Benchmark Notebook

Confronto tra:
- Deployment vLLM
- Deployment llm-d + vLLM

Metriche:
- Time To First Token (TTFT)
- Latenza totale
- Token output
- Throughput (req/sec)
- Throughput (token/sec)

Compila gli endpoint nelle celle successive.


In [ ]:
# Configurazione

VLLM_URL = "http://vllm-endpoint/v1/chat/completions"
LLMD_URL = "http://llmd-endpoint/v1/chat/completions"

MODEL_NAME = "qwen3-32b"

CONCURRENT_USERS = [1, 10, 25, 50, 100]
REQUESTS_PER_TEST = 100
MAX_TOKENS = 256


In [ ]:
import requests
import time
import threading
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd


In [ ]:
PROMPT = '''
Explain confidential computing on OpenShift and describe how attestation,
trusted execution environments and GPU acceleration work together.
'''


In [ ]:
def send_request(endpoint):

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": PROMPT}
        ],
        "max_tokens": MAX_TOKENS,
        "stream": False
    }

    start = time.time()

    r = requests.post(endpoint, json=payload, timeout=300)

    end = time.time()

    latency = end - start

    try:
        data = r.json()
        completion_tokens = data.get("usage", {}).get("completion_tokens", 0)
    except Exception:
        completion_tokens = 0

    return {
        "status": r.status_code,
        "latency": latency,
        "completion_tokens": completion_tokens
    }


In [ ]:
def benchmark(endpoint, users, total_requests):

    results = []

    start = time.time()

    with ThreadPoolExecutor(max_workers=users) as executor:

        futures = [
            executor.submit(send_request, endpoint)
            for _ in range(total_requests)
        ]

        for f in as_completed(futures):
            results.append(f.result())

    duration = time.time() - start

    successful = [r for r in results if r["status"] == 200]

    latencies = [r["latency"] for r in successful]

    tokens = sum(r["completion_tokens"] for r in successful)

    return {
        "users": users,
        "requests": total_requests,
        "success": len(successful),
        "errors": total_requests - len(successful),
        "avg_latency": statistics.mean(latencies) if latencies else None,
        "p95_latency": sorted(latencies)[int(len(latencies)*0.95)] if latencies else None,
        "throughput_req_s": total_requests / duration,
        "throughput_tok_s": tokens / duration if duration > 0 else 0
    }


In [ ]:
def run_suite(endpoint, label):

    rows = []

    for users in CONCURRENT_USERS:

        print(f"Running {label} - users={users}")

        rows.append(
            benchmark(
                endpoint,
                users,
                REQUESTS_PER_TEST
            )
        )

    return pd.DataFrame(rows)


In [ ]:
vllm_df = run_suite(VLLM_URL, "vLLM")
vllm_df


In [ ]:
llmd_df = run_suite(LLMD_URL, "llm-d")
llmd_df


In [ ]:
comparison = vllm_df.merge(
    llmd_df,
    on="users",
    suffixes=("_vllm", "_llmd")
)

comparison


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    comparison["users"],
    comparison["throughput_tok_s_vllm"],
    marker="o",
    label="vLLM"
)

plt.plot(
    comparison["users"],
    comparison["throughput_tok_s_llmd"],
    marker="o",
    label="llm-d"
)

plt.xlabel("Concurrent users")
plt.ylabel("Token/sec")
plt.legend()
plt.grid(True)

plt.show()


In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    comparison["users"],
    comparison["p95_latency_vllm"],
    marker="o",
    label="vLLM"
)

plt.plot(
    comparison["users"],
    comparison["p95_latency_llmd"],
    marker="o",
    label="llm-d"
)

plt.xlabel("Concurrent users")
plt.ylabel("P95 latency (s)")
plt.legend()
plt.grid(True)

plt.show()
